# LeetCode 1877 — Minimize Maximum Pair Sum in Array bằng Python

Link đề: https://leetcode.com/problems/minimize-maximum-pair-sum-in-array/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def minPairSum(self, nums: List[int]) -> int:
        ...
```

Lộ trình:
1. **Vét cạn mọi cách chia cặp** (đệ quy, `(n−1)!!` cách) — đáp án chuẩn cho `n ≤ 12`.
2. **Tìm kiếm nhị phân trên đáp án** + kiểm tra tham lam "có chia được sao cho mọi tổng ≤ X không?" (`O(n log n + n log V)`).
3. **Sắp xếp + ghép nhỏ nhất với lớn nhất** (hai con trỏ) — `O(n log n)`, chứng minh bằng lập luận đổi chỗ.
4. **Counting sort** (giá trị `≤ 10⁵`) + hai con trỏ trên bảng đếm — `O(n + V)`.

## 1. Mục tiêu học

1. Hiểu bài toán **minimax**: tối thiểu hóa giá trị **lớn nhất** trong các tổng cặp — khác với tối thiểu hóa tổng.
2. Đề xuất và **chứng minh** chiến lược tham lam "nhỏ nhất ghép lớn nhất" bằng **lập luận đổi chỗ** và bằng **cận dưới** `max_i (a_i + a_{n−1−i})`.
3. Biết kỹ thuật tổng quát **tìm kiếm nhị phân trên đáp án** cho bài "minimize the maximum", kèm hàm kiểm tra khả thi.
4. Cài **hai con trỏ** trên mảng đã sắp và trên **bảng đếm** (counting sort) khi miền giá trị nhỏ.
5. Kiểm chứng tham lam bằng vét cạn mọi cách chia cặp trên input ngẫu nhiên.

## 2. Đề bài & ví dụ

**Tổng cặp** của cặp `(a, b)` là `a + b`. **Tổng cặp lớn nhất** của một danh sách cặp là giá trị lớn nhất trong các tổng cặp.
Ví dụ các cặp `(1,5), (2,3), (4,4)` có tổng cặp lớn nhất `max(6, 5, 8) = 8`.

Cho mảng `nums` có độ dài `n` **chẵn**. Chia các phần tử thành `n/2` cặp sao cho:
- mỗi phần tử thuộc đúng một cặp, và
- **tổng cặp lớn nhất là nhỏ nhất**.

Trả về giá trị nhỏ nhất đó.

**Ví dụ 1:** `nums = [3,5,2,3]` → `7`. Ghép `(3,3)` và `(5,2)` ⇒ `max(6, 7) = 7`.

**Ví dụ 2:** `nums = [3,5,4,2,4,6]` → `8`. Ghép `(3,5), (4,4), (6,2)` ⇒ `max(8, 8, 8) = 8`.

**Ràng buộc:**
- `n == nums.length`, `2 ≤ n ≤ 10⁵`, `n` chẵn
- `1 ≤ nums[i] ≤ 10⁵`

In [1]:
import random
import time
from bisect import bisect_left
from typing import List

examples = [
    ([3, 5, 2, 3], 7),
    ([3, 5, 4, 2, 4, 6], 8),
]
for nums, want in examples:
    print(f"nums={nums} -> kỳ vọng {want}")

nums=[3, 5, 2, 3] -> kỳ vọng 7
nums=[3, 5, 4, 2, 4, 6] -> kỳ vọng 8


## 3. Kiến thức nền

### 3.1 Sắp xếp và hai con trỏ đối đầu

Sau khi sắp, con trỏ `l = 0` chỉ phần tử nhỏ nhất, `r = n − 1` chỉ phần tử lớn nhất; mỗi bước `l += 1`, `r −= 1` tới khi gặp nhau. Tổng cộng `n/2` bước.

In [2]:
a = sorted([3, 5, 4, 2, 4, 6])
l, r = 0, len(a) - 1
while l < r:
    print(f"cặp (a[{l}]={a[l]}, a[{r}]={a[r]}) tổng {a[l] + a[r]}")
    l += 1; r -= 1

cặp (a[0]=2, a[5]=6) tổng 8
cặp (a[1]=3, a[4]=5) tổng 8
cặp (a[2]=4, a[3]=4) tổng 8


### 3.2 Biểu thức sinh với `zip` và slicing

`zip(a, reversed(a))` ghép `a[0]` với `a[n−1]`, `a[1]` với `a[n−2]`, … Chỉ cần nửa đầu: `zip(a[:n//2], a[::-1])`.

In [3]:
a = sorted([3, 5, 2, 3])
print(list(zip(a[:len(a) // 2], a[::-1])), max(x + y for x, y in zip(a[:len(a) // 2], a[::-1])))

[(2, 5), (3, 3)] 7


### 3.3 Tìm kiếm nhị phân trên đáp án

Nếu tồn tại cách chia với mọi tổng `≤ X` thì cũng tồn tại với mọi `X' > X` (tính **đơn điệu**). Vậy có thể tìm `X` nhỏ nhất khả thi bằng nhị phân trên đoạn `[min tổng có thể, max tổng có thể]`.

In [4]:
def first_true(lo, hi, pred):
    """X nhỏ nhất trong [lo, hi] với pred(X) đúng (giả sử pred đơn điệu và pred(hi) đúng)."""
    while lo < hi:
        mid = (lo + hi) // 2
        if pred(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print(first_true(0, 100, lambda x: x * x >= 50))

8


### 3.4 Bảng đếm cho giá trị nhỏ

`cnt[v]` = số lần `v` xuất hiện. Hai con trỏ `i` (giá trị nhỏ nhất còn lại) và `j` (lớn nhất còn lại) đi trên **giá trị**, không phải chỉ số.

In [5]:
cnt = [0] * 8
for v in [3, 5, 4, 2, 4, 6]:
    cnt[v] += 1
print({v: c for v, c in enumerate(cnt) if c})

{2: 1, 3: 1, 4: 2, 5: 1, 6: 1}


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 `[3,5,2,3]` có `3` cách chia cặp:

| cách chia | tổng các cặp | max |
|---|---|---|
| (3,5), (2,3) | 8, 5 | 8 |
| (3,2), (5,3) | 5, 8 | 8 |
| (3,3), (5,2) | 6, 7 | **7** |

Cách tốt nhất ghép **lớn nhất (5) với nhỏ nhất (2)**. Trực giác: số lớn nhất `M` phải ghép với ai đó; ghép nó với số nhỏ nhất `m` cho tổng `M + m` nhỏ nhất có thể cho cặp chứa `M`, đồng thời "giải phóng" các số lớn khác khỏi việc phải gặp `M`.

**Cận dưới đẹp:** sau khi sắp `a₀ ≤ … ≤ a_{n−1}`, với mỗi `i < n/2`, mọi cách chia đều có max `≥ a_i + a_{n−1−i}` (nguyên lý chuồng bồ câu — chứng minh ở mục 10). Cách ghép đối đầu đạt đúng `max_i (a_i + a_{n−1−i})` ⇒ tối ưu.

In [6]:
def all_pairings(arr):
    """Sinh mọi cách chia arr (độ dài chẵn) thành các cặp."""
    if not arr:
        yield []
        return
    first = arr[0]
    for k in range(1, len(arr)):
        rest = arr[1:k] + arr[k + 1:]
        for p in all_pairings(rest):
            yield [(first, arr[k])] + p

for p in all_pairings([3, 5, 2, 3]):
    print(p, "-> max tổng =", max(x + y for x, y in p))

[(3, 5), (2, 3)] -> max tổng = 8
[(3, 2), (5, 3)] -> max tổng = 8
[(3, 3), (5, 2)] -> max tổng = 7


## 5. Cách 1 — Brute Force / baseline: thử mọi cách chia cặp

**Ý tưởng:** liệt kê mọi cách chia (phần tử đầu ghép với mỗi phần tử còn lại, đệ quy phần còn lại), tính max tổng, lấy min. Số cách chia là `(n−1)!! = (n−1)(n−3)…1`.

In [7]:
def min_pair_sum_brute(nums):
    best = None
    for p in all_pairings(list(nums)):
        m = max(x + y for x, y in p)
        if best is None or m < best:
            best = m
    return best

for nums, want in examples:
    print(min_pair_sum_brute(nums), "kỳ vọng", want)

7 kỳ vọng 7
8 kỳ vọng 8


Có thể cắt tỉa: nếu max hiện tại của phần đã ghép đã `≥ best` thì bỏ nhánh. Bản dưới thêm cắt tỉa, vẫn là vét cạn.

In [8]:
def min_pair_sum_brute_pruned(nums):
    arr = list(nums); best = [float("inf")]
    def go(rest, cur_max):
        if cur_max >= best[0]:
            return
        if not rest:
            best[0] = cur_max; return
        first = rest[0]
        for k in range(1, len(rest)):
            go(rest[1:k] + rest[k + 1:], max(cur_max, first + rest[k]))
    go(arr, 0)
    return best[0]

for n in [8, 10, 12]:
    random.seed(n)
    nums = [random.randint(1, 100) for _ in range(n)]
    t0 = time.perf_counter(); a = min_pair_sum_brute(nums); t1 = time.perf_counter(); b = min_pair_sum_brute_pruned(nums)
    print(f"n={n}: {a} ({t1 - t0:.3f}s) | có cắt tỉa: {b} ({time.perf_counter() - t1:.3f}s)")

n=8: 97 (0.000s) | có cắt tỉa: 97 (0.000s)
n=10: 115 (0.001s) | có cắt tỉa: 115 (0.000s)
n=12: 106 (0.012s) | có cắt tỉa: 106 (0.000s)


**Độ phức tạp:** `(n−1)!!` cách chia × `O(n)` ⇒ siêu mũ. `n = 12` có `10395` cách, `n = 20` có `654 729 075` cách.

**Nút thắt:** ta không tận dụng cấu trúc thứ tự. Sau khi sắp, chỉ có **một** cách ghép "hợp lý".

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (đổi chỗ).** Sắp tăng `a`. Giả sử trong một cách chia tối ưu, `a_{n−1}` (lớn nhất) ghép với `x`, còn `a₀` (nhỏ nhất) ghép với `y`. Đổi thành `(a_{n−1}, a₀)` và `(x, y)`:
- `a_{n−1} + a₀ ≤ a_{n−1} + x` (vì `a₀ ≤ x`),
- `x + y ≤ a_{n−1} + x` (vì `y ≤ a_{n−1}`).

Vậy hai cặp mới đều `≤ a_{n−1} + x` (tổng cặp cũ chứa max) ⇒ max không tăng. Lặp lại cho mảng còn lại ⇒ ghép `a_i` với `a_{n−1−i}`.

**Insight 2 (nhị phân trên đáp án).** Với ngưỡng `X`, kiểm tra khả thi bằng tham lam: phần tử lớn nhất còn lại phải ghép với **phần tử nhỏ nhất** còn lại (nếu ngay cả phần tử nhỏ nhất cũng làm tổng `> X` thì không khả thi). Đơn điệu theo `X` ⇒ tìm nhị phân.

**Quy tắc:** sắp tăng, trả về `max_{i < n/2} (a_i + a_{n−1−i})`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tìm kiếm nhị phân trên đáp án

Từng đoạn:
1. Sắp `a`. Miền tìm kiếm: `[a₀ + a₁, a_{n−2} + a_{n−1}]` (tổng nhỏ nhất và lớn nhất có thể của một cặp).
2. `feasible(X)`: hai con trỏ `l = 0`, `r = n − 1`; nếu `a[l] + a[r] > X` thì không khả thi (phần tử lớn nhất `a[r]` không ghép được với ai); ngược lại ghép, `l += 1`, `r −= 1`.
3. Tìm `X` nhỏ nhất khả thi.

In [9]:
def min_pair_sum_binary(nums):
    a = sorted(nums)
    n = len(a)

    def feasible(X):
        l, r = 0, n - 1
        while l < r:
            if a[l] + a[r] > X:
                return False
            l += 1
            r -= 1
        return True

    lo, hi = a[0] + a[1], a[-2] + a[-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

for nums, want in examples:
    print(min_pair_sum_binary(nums), "kỳ vọng", want)

7 kỳ vọng 7
8 kỳ vọng 8


**Độ phức tạp:** sắp `O(n log n)` + `log(2·10⁵)` lần kiểm tra `O(n)` ⇒ `O(n log n + n log V)`. Space `O(n)`.

Cách này "thừa" cho bài này (vì `feasible` đã ngầm dùng đúng cặp của Cách 3), nhưng là **khuôn mẫu chung** cho mọi bài "minimize the maximum" mà chưa nghĩ ra tham lam trực tiếp.

### 7.2 Cách 3 — Sắp xếp + ghép nhỏ nhất với lớn nhất

In [10]:
def min_pair_sum_sort(nums):
    a = sorted(nums)
    n = len(a)
    return max(a[i] + a[n - 1 - i] for i in range(n // 2))

for nums, want in examples:
    print(min_pair_sum_sort(nums), "kỳ vọng", want)

7 kỳ vọng 7
8 kỳ vọng 8


Code nộp LeetCode của Cách 3:

```python
class Solution:
    def minPairSum(self, nums: List[int]) -> int:
        a = sorted(nums)
        n = len(a)
        return max(a[i] + a[n - 1 - i] for i in range(n // 2))
```

**Độ phức tạp:** Time `O(n log n)`, Space `O(n)` (hoặc `O(1)` phụ nếu `nums.sort()` tại chỗ).

### 7.3 Cách 4 — Counting sort + hai con trỏ trên giá trị

Từng đoạn:
1. `cnt[v]` cho `v ∈ [1, 10⁵]`.
2. `i` = giá trị nhỏ nhất còn lại, `j` = giá trị lớn nhất còn lại.
3. Nếu `i == j`: các phần tử còn lại đều bằng `i`, ghép với nhau ⇒ tổng `2i`; dừng.
4. Ngược lại ghép `t = min(cnt[i], cnt[j])` cặp `(i, j)`, cập nhật `best = max(best, i + j)`.

In [11]:
def min_pair_sum_count(nums, V=10**5):
    cnt = [0] * (V + 2)
    lo_v, hi_v = V, 1
    for x in nums:
        cnt[x] += 1
        if x < lo_v: lo_v = x
        if x > hi_v: hi_v = x
    i, j = lo_v, hi_v
    best = 0
    while True:
        while i <= j and cnt[i] == 0:
            i += 1
        while j >= i and cnt[j] == 0:
            j -= 1
        if i > j:
            break
        if i == j:
            best = max(best, 2 * i)        # cnt[i] chẵn, ghép với nhau
            break
        t = min(cnt[i], cnt[j])
        best = max(best, i + j)
        cnt[i] -= t
        cnt[j] -= t
    return best

for nums, want in examples:
    print(min_pair_sum_count(nums), "kỳ vọng", want)

7 kỳ vọng 7
8 kỳ vọng 8


Khi `i == j`, số lượng còn lại chẵn vì tổng số phần tử chẵn và mỗi bước ta trừ cùng `t` ở hai giá trị khác nhau (bớt `2t` phần tử). Chỉ xét đoạn `[min(nums), max(nums)]` để không quét cả `10⁵` ô với input nhỏ.

**Độ phức tạp:** `O(n + (max − min))`, Space `O(V)`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2 `[3,5,4,2,4,6]` → sắp `[2,3,4,4,5,6]`:

| i | a[i] | a[n−1−i] | tổng | max hiện tại |
|---|---|---|---|---|
| 0 | 2 | 6 | 8 | 8 |
| 1 | 3 | 5 | 8 | 8 |
| 2 | 4 | 4 | 8 | **8** |

In [12]:
def trace_sort(nums):
    a = sorted(nums); n = len(a); best = 0
    print("sau khi sắp:", a)
    for i in range(n // 2):
        s = a[i] + a[n - 1 - i]; best = max(best, s)
        print(f"i={i}: ghép {a[i]} với {a[n - 1 - i]} -> tổng {s}, max = {best}")
    return best

trace_sort([3, 5, 4, 2, 4, 6])

sau khi sắp: [2, 3, 4, 4, 5, 6]
i=0: ghép 2 với 6 -> tổng 8, max = 8
i=1: ghép 3 với 5 -> tổng 8, max = 8
i=2: ghép 4 với 4 -> tổng 8, max = 8


8

In [13]:
def trace_binary(nums):
    a = sorted(nums); n = len(a)
    def feasible(X):
        l, r = 0, n - 1
        while l < r:
            if a[l] + a[r] > X: return False
            l += 1; r -= 1
        return True
    lo, hi = a[0] + a[1], a[-2] + a[-1]
    print(f"miền tìm kiếm [{lo}, {hi}]")
    while lo < hi:
        mid = (lo + hi) // 2; ok = feasible(mid)
        print(f"  thử X={mid}: {'khả thi' if ok else 'không khả thi'} -> ", end="")
        if ok: hi = mid
        else: lo = mid + 1
        print(f"[{lo}, {hi}]")
    print("đáp án:", lo)
    return lo

trace_binary([3, 5, 2, 3])
trace_binary([1, 9, 2, 8, 3, 7, 4, 6])

miền tìm kiếm [5, 8]
  thử X=6: không khả thi -> [7, 8]
  thử X=7: khả thi -> [7, 7]
đáp án: 7
miền tìm kiếm [3, 17]
  thử X=10: khả thi -> [3, 10]
  thử X=6: không khả thi -> [7, 10]
  thử X=8: không khả thi -> [9, 10]
  thử X=9: không khả thi -> [10, 10]
đáp án: 10


10

In [14]:
def trace_count(nums):
    V = max(nums); cnt = [0] * (V + 2)
    for x in nums: cnt[x] += 1
    i, j, best = min(nums), V, 0
    while True:
        while i <= j and cnt[i] == 0: i += 1
        while j >= i and cnt[j] == 0: j -= 1
        if i > j: break
        if i == j:
            best = max(best, 2 * i); print(f"còn {cnt[i]} phần tử bằng {i}, ghép với nhau -> tổng {2 * i}"); break
        t = min(cnt[i], cnt[j]); best = max(best, i + j)
        print(f"ghép {t} cặp ({i}, {j}) tổng {i + j}, max = {best}")
        cnt[i] -= t; cnt[j] -= t
    return best

trace_count([3, 5, 4, 2, 4, 6, 1, 1])

ghép 1 cặp (1, 6) tổng 7, max = 7
ghép 1 cặp (1, 5) tổng 6, max = 7
ghép 1 cặp (2, 4) tổng 6, max = 7
ghép 1 cặp (3, 4) tổng 7, max = 7


7

### 8.1 Trace lập luận đổi chỗ

Bắt đầu từ một cách chia **bất kỳ**, lặp lại: tìm cặp chứa phần tử lớn nhất còn "chưa cố định" và cặp chứa phần tử nhỏ nhất, đổi bạn của chúng cho nhau (mục 6). In max tổng cặp sau mỗi lần đổi — nó **không bao giờ tăng**, và cuối cùng ta được cách ghép đối đầu.

In [15]:
def trace_exchange(pairs):
    pairs = [tuple(sorted(p)) for p in pairs]
    fixed = []
    step = 0
    print(f"ban đầu: {pairs} max = {max(x + y for x, y in pairs)}")
    while pairs:
        elems = sorted(x for p in pairs for x in p)
        lo, hi = elems[0], elems[-1]
        p_hi = next(p for p in pairs if hi in p)
        if lo in p_hi:                              # lớn nhất đã ghép với nhỏ nhất
            fixed.append(p_hi); pairs.remove(p_hi); continue
        p_lo = next(p for p in pairs if lo in p)
        x = p_hi[0] if p_hi[1] == hi else p_hi[1]   # bạn của hi
        y = p_lo[0] if p_lo[1] == lo else p_lo[1]   # bạn của lo
        pairs.remove(p_hi); pairs.remove(p_lo)
        pairs += [(lo, hi), tuple(sorted((x, y)))]
        step += 1
        allp = fixed + pairs
        print(f"đổi {step}: ({x},{hi}) & ({lo},{y}) -> ({lo},{hi}) & ({min(x, y)},{max(x, y)}) | max = {max(a + b for a, b in allp)}")
    print("kết quả:", sorted(fixed))

trace_exchange([(5, 6), (2, 3), (4, 4)])
print()
trace_exchange([(1, 2), (3, 4), (5, 6), (7, 8)])   # ghép kề nhau -> đối đầu

ban đầu: [(5, 6), (2, 3), (4, 4)] max = 11
đổi 1: (5,6) & (2,3) -> (2,6) & (3,5) | max = 8
kết quả: [(2, 6), (3, 5), (4, 4)]

ban đầu: [(1, 2), (3, 4), (5, 6), (7, 8)] max = 15
đổi 1: (7,8) & (1,2) -> (1,8) & (2,7) | max = 11
đổi 2: (5,6) & (3,4) -> (3,6) & (4,5) | max = 9
kết quả: [(1, 8), (2, 7), (3, 6), (4, 5)]


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def minPairSum(self, nums: List[int]) -> int:
        nums = sorted(nums)
        n = len(nums)
        best = 0
        for i in range(n // 2):
            s = nums[i] + nums[n - 1 - i]
            if s > best:
                best = s
        return best

sol = Solution()
for nums, want in examples:
    got = sol.minPairSum(nums)
    print(got, "OK" if got == want else "SAI")

7 OK
8 OK


## 10. Vì sao thuật toán đúng

Sắp `a₀ ≤ a₁ ≤ … ≤ a_{n−1}`, gọi `G = max_{i < n/2} (a_i + a_{n−1−i})` (giá trị của cách ghép tham lam).

**(a) Cận dưới: mọi cách chia có max `≥ G`.** Cố định `i < n/2`. Xét `i + 1` phần tử lớn nhất `L = {a_{n−1−i}, …, a_{n−1}}` (theo vị trí sau khi sắp) và `i` phần tử nhỏ nhất `S = {a₀, …, a_{i−1}}`. Trong một cách chia bất kỳ, mỗi phần tử của `L` có một bạn ghép. Vì `|S| = i < i + 1 = |L|`, **không thể** mọi phần tử của `L` đều có bạn trong `S` (mỗi phần tử của `S` chỉ làm bạn của một người). Vậy có `a_q ∈ L` (tức `q ≥ n − 1 − i`) với bạn `a_p` có chỉ số `p ≥ i` (bạn có thể nằm trong `L` hoặc ở giữa — đều có chỉ số `≥ i`). Tổng cặp đó `a_p + a_q ≥ a_i + a_{n−1−i}`. Do đó max của cách chia `≥ a_i + a_{n−1−i}` với mọi `i` ⇒ max `≥ G`.

**(b) Tham lam đạt `G`.** Hiển nhiên — max của các cặp tham lam chính là `G`.

Từ (a) và (b), `G` là tối ưu. (Lập luận đổi chỗ ở mục 6 là một chứng minh khác.)

**Cách 2 đúng** vì `feasible(X)` đúng ⇔ `G ≤ X`: nếu cặp tham lam nào có tổng `> X` thì theo (a) mọi cách chia đều có max `> X`.

## 11. Phân tích Time / Space

Gọi `n ≤ 10⁵`, `V = 10⁵`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn chia cặp | `O((n−1)!! · n)` | `O(n)` | `n ≤ 12` |
| 2. Nhị phân trên đáp án | `O(n log n + n log V)` | `O(n)` | khuôn mẫu chung |
| 3. Sort + hai con trỏ | `O(n log n)` | `O(n)` | **nộp bài** |
| 4. Counting + hai con trỏ | `O(n + V)` | `O(V)` | khi giá trị nhỏ |

In [17]:
random.seed(1)
nums = [random.randint(1, 10**5) for _ in range(10**5)]
for name, fn in [("binary", min_pair_sum_binary), ("sort", min_pair_sum_sort), ("count", min_pair_sum_count),
                 ("submit", Solution().minPairSum)]:
    t0 = time.perf_counter(); r = fn(nums)
    print(f"{name:6s}: {r}  {time.perf_counter() - t0:.4f}s")

binary: 100244  0.1088s


sort  : 100244  0.0200s


count : 100244  0.0164s


submit: 100244  0.0197s


## 12. Bộ test

In [18]:
fns = [min_pair_sum_brute_pruned, min_pair_sum_binary, min_pair_sum_sort, min_pair_sum_count,
       lambda a: Solution().minPairSum(a)]

for nums, want in examples:
    assert min_pair_sum_brute(nums) == want
    for fn in fns:
        assert fn(nums) == want

edge = [([1, 1], 2), ([1, 100000], 100001), ([5, 5, 5, 5], 10), ([1, 2, 3, 4], 5),
        ([1, 1, 1, 100], 101), ([100, 100, 1, 1], 101), ([1, 2, 99, 100], 101)]
for nums, want in edge:
    assert min_pair_sum_brute(nums) == want, nums
    for fn in fns:
        assert fn(nums) == want, (fn, nums)

random.seed(2024)
for _ in range(600):
    n = random.choice([2, 4, 6, 8, 10])
    hi = random.choice([3, 10, 100, 10**5])
    nums = [random.randint(1, hi) for _ in range(n)]
    want = min_pair_sum_brute(nums)
    for fn in fns:
        assert fn(nums) == want, (fn, nums)
print("ví dụ + edge + 600 bộ ngẫu nhiên khớp vét cạn mọi cách chia cặp")

ví dụ + edge + 600 bộ ngẫu nhiên khớp vét cạn mọi cách chia cặp


In [19]:
random.seed(7)
for _ in range(100):
    n = 2 * random.randint(1, 2000)
    nums = [random.randint(1, random.choice([5, 1000, 10**5])) for _ in range(n)]
    r = min_pair_sum_sort(nums)
    assert r == min_pair_sum_binary(nums) == min_pair_sum_count(nums)
    # kiểm tra cận dưới: không cặp tham lam nào vượt r, và có cặp đạt r
    a = sorted(nums)
    assert max(a[i] + a[-1 - i] for i in range(n // 2)) == r
print("n tới 4000: ba cách nhanh khớp nhau")

nums = [3, 5, 2, 3]
Solution().minPairSum(nums)
assert nums == [3, 5, 2, 3]
print("input không bị sửa")

n tới 4000: ba cách nhanh khớp nhau
input không bị sửa


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Ghép các phần tử **kề nhau** sau khi sắp

In [20]:
def wrong_adjacent(nums):
    a = sorted(nums)
    return max(a[i] + a[i + 1] for i in range(0, len(a), 2))

for nums, want in examples:
    print("sai =", wrong_adjacent(nums), "| đúng =", want)

sai = 8 | đúng = 7
sai = 11 | đúng = 8


Ghép kề nhau làm hai số lớn nhất gặp nhau ⇒ max lớn. (Ghép kề nhau là chiến lược đúng cho bài 561 *Array Partition* — **tối đa tổng các min** — không phải bài này.) **Sửa:** ghép đối đầu.

### Lỗi 2 — Trả về `max + min` (chỉ xét cặp đầu tiên)

In [21]:
def wrong_first_pair(nums):
    return max(nums) + min(nums)

for nums, want in examples + [([1, 9, 5, 5], 10), ([1, 2, 9, 10], 11), ([4, 4, 4, 1], 8)]:
    print(f"{nums}: sai = {wrong_first_pair(nums)} | đúng = {min_pair_sum_sort(nums)}")

[3, 5, 2, 3]: sai = 7 | đúng = 7
[3, 5, 4, 2, 4, 6]: sai = 8 | đúng = 8
[1, 9, 5, 5]: sai = 10 | đúng = 10
[1, 2, 9, 10]: sai = 11 | đúng = 11
[4, 4, 4, 1]: sai = 5 | đúng = 8


`[4,4,4,1]`: cặp `(1,4)` = 5 nhưng cặp còn lại `(4,4)` = 8. **Sửa:** lấy max trên **mọi** cặp đối đầu.

### Lỗi 3 — Không sắp xếp, ghép `nums[i]` với `nums[n−1−i]` theo thứ tự gốc

In [22]:
def wrong_no_sort(nums):
    n = len(nums)
    return max(nums[i] + nums[n - 1 - i] for i in range(n // 2))

for nums, want in examples:
    print("sai =", wrong_no_sort(nums), "| đúng =", want)

sai = 7 | đúng = 7
sai = 9 | đúng = 8


**Sửa:** sắp trước khi ghép.

### Lỗi 4 — Tìm nhị phân sai biên (`hi` quá nhỏ hoặc cập nhật `hi = mid − 1`)

In [23]:
def wrong_binary(nums):
    a = sorted(nums); n = len(a)
    def feasible(X):
        return all(a[i] + a[n - 1 - i] <= X for i in range(n // 2))
    lo, hi = a[0] + a[1], a[-2] + a[-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(mid):
            hi = mid - 1                       # SAI: có thể bỏ lỡ chính mid
        else:
            lo = mid + 1
    return lo

random.seed(4)
for _ in range(5000):
    nums = [random.randint(1, 20) for _ in range(random.choice([2, 4, 6]))]
    if wrong_binary(nums) != min_pair_sum_sort(nums):
        print(f"phản ví dụ: {nums} | sai = {wrong_binary(nums)} | đúng = {min_pair_sum_sort(nums)}")
        break

phản ví dụ: [18, 12, 9, 6, 4, 9] | sai = 21 | đúng = 22


Với khuôn "tìm `X` nhỏ nhất thỏa", khi `feasible(mid)` đúng phải giữ `hi = mid` (mid có thể là đáp án). **Sửa:** `hi = mid`, `lo = mid + 1`, vòng `while lo < hi`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chia thành cặp, **tối thiểu max** tổng cặp" ⇒ sắp xếp, ghép **nhỏ nhất với lớn nhất**.
- "Chia thành cặp, **tối đa tổng các min**" (561) ⇒ ghép **kề nhau**.
- "Minimize the maximum" chưa rõ tham lam ⇒ **nhị phân trên đáp án** + kiểm tra tham lam.
- Chứng minh tham lam cặp: lập luận đổi chỗ hai cặp, hoặc cận dưới bằng chuồng bồ câu.
- Họ bài: 881 *Boats to Save People* (ghép nặng nhất với nhẹ nhất nếu vừa), 2491 *Divide Players Into Teams of Equal Skill*, 1679 *Max Number of K-Sum Pairs*.

## 15. Biến thể của bài

1. **Tối đa hóa min tổng cặp:** cũng ghép nhỏ–lớn (lập luận đối xứng).
2. **Chia thành nhóm 3 (hoặc `k`), tối thiểu max tổng nhóm:** không còn công thức đơn giản — NP-khó nói chung (bài toán phân hoạch); với `k` nhỏ có heuristic.
3. **881 Boats to Save People:** mỗi thuyền tối đa 2 người, tổng `≤ limit`, tối thiểu số thuyền ⇒ hai con trỏ: người nặng nhất luôn lên thuyền, ghép thêm người nhẹ nhất nếu vừa.
4. **Trả về chính cách chia:** cặp `(a_i, a_{n−1−i})`.
5. **Tối thiểu hiệu giữa tổng cặp lớn nhất và nhỏ nhất:** ghép nhỏ–lớn cũng tối ưu? — xem Bài 5.

In [24]:
def max_min_pair_sum(nums):
    """Biến thể 1: tối đa hóa min tổng cặp."""
    a = sorted(nums); n = len(a)
    return min(a[i] + a[n - 1 - i] for i in range(n // 2))

def brute_max_min(nums):
    return max(min(x + y for x, y in p) for p in all_pairings(list(nums)))

def num_rescue_boats(people, limit):
    """Biến thể 3 (LeetCode 881)."""
    a = sorted(people); l, r = 0, len(a) - 1; boats = 0
    while l <= r:
        if a[l] + a[r] <= limit:
            l += 1
        r -= 1; boats += 1
    return boats

random.seed(3)
for _ in range(300):
    nums = [random.randint(1, 30) for _ in range(random.choice([2, 4, 6, 8]))]
    assert max_min_pair_sum(nums) == brute_max_min(nums)
print("biến thể 1 khớp vét cạn;", "881 ví dụ:", num_rescue_boats([3, 2, 2, 1], 3), num_rescue_boats([3, 5, 3, 4], 5))

biến thể 1 khớp vét cạn; 881 ví dụ: 3 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi cách chia cặp | siêu mũ | `O(n)` | kiểm chứng |
| 2. Nhị phân đáp án | `feasible(X)` tham lam | `O(n log n + n log V)` | `O(n)` | khuôn "minimize max" |
| 3. Sort + đối đầu | nhỏ ghép lớn | `O(n log n)` | `O(n)` | **nộp bài** |
| 4. Counting | hai con trỏ trên giá trị | `O(n + V)` | `O(V)` | giá trị nhỏ |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `minPairSum([1, 2, 3, 4, 5, 6])` và `minPairSum([10, 10, 1, 1, 1, 1])`?

**Bài 2 (tính tay):** liệt kê 15 cách chia cặp của `[1, 4, 6, 7, 9, 10]`? (Không cần hết — chỉ ra cận dưới `max(a_i + a_{n−1−i})` và một cách đạt được.)

**Bài 3 (điền TODO):** hoàn thiện `optimal_pairs(nums)` trả về danh sách cặp của một cách chia tối ưu.

**Bài 4 (tự viết lại):** viết lại Cách 3 với `nums.sort()` tại chỗ trên **bản sao**, và chỉ dùng vòng `while l < r`.

**Bài 5 (mini challenge):** tối thiểu hóa **(tổng cặp lớn nhất − tổng cặp nhỏ nhất)**. Ghép nhỏ–lớn có luôn tối ưu không? Kiểm chứng bằng vét cạn (in phản ví dụ nếu có) và thử chứng minh. (Biến thể tự đặt.)

In [25]:
def optimal_pairs(nums):
    # TODO: sắp, ghép a[i] với a[n-1-i]
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Sắp: `[1..6]`. (2) Các cặp `(1,6), (2,5), (3,4)` đều tổng 7. (3) `[1,1,1,1,10,10]` ⇒ cặp `(1,10), (1,10), (1,1)` ⇒ `11`.

**Bài 2:** (1) Sắp `[1,4,6,7,9,10]`. (2) Tổng đối đầu: `11, 13, 13` ⇒ cận dưới 13. (3) Cách `(1,10), (4,9), (6,7)` đạt 13.

**Bài 3:** (1) `a = sorted(nums)`. (2) `[(a[i], a[n−1−i]) for i in range(n//2)]`. (3) Kiểm tra `max` tổng bằng đáp án.

**Bài 4:** (1) `a = list(nums); a.sort()`. (2) `l, r = 0, n − 1`. (3) Cập nhật `best` rồi dịch hai con trỏ.

**Bài 5:** (1) Viết hàm khoảng cách `max − min` cho một cách chia. (2) Vét cạn với `n ≤ 8`. (3) So với ghép nhỏ–lớn trên vài nghìn input ngẫu nhiên.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
assert Solution().minPairSum([1, 2, 3, 4, 5, 6]) == 7 and Solution().minPairSum([10, 10, 1, 1, 1, 1]) == 11

# Bài 2
a = [1, 4, 6, 7, 9, 10]
assert min_pair_sum_brute(a) == 13 == max(a[i] + a[-1 - i] for i in range(3))

# Bài 3
def optimal_pairs_ans(nums):
    a = sorted(nums); n = len(a)
    return [(a[i], a[n - 1 - i]) for i in range(n // 2)]

# Bài 4
def min_pair_sum_while(nums):
    a = list(nums); a.sort(); l, r = 0, len(a) - 1; best = 0
    while l < r:
        best = max(best, a[l] + a[r]); l += 1; r -= 1
    return best

random.seed(111)
for _ in range(500):
    nums = [random.randint(1, 50) for _ in range(random.choice([2, 4, 6, 8]))]
    pairs = optimal_pairs_ans(nums)
    assert sorted(x for p in pairs for x in p) == sorted(nums)
    assert max(x + y for x, y in pairs) == min_pair_sum_brute(nums) == min_pair_sum_while(nums)
print("Bài 1-4: OK")

Bài 1-4: OK


In [27]:
# Bài 5 — tối thiểu (max tổng cặp - min tổng cặp)
def spread(p):
    sums = [x + y for x, y in p]
    return max(sums) - min(sums)

def brute_spread(nums):
    return min(spread(p) for p in all_pairings(list(nums)))

random.seed(112)
counter = None
for _ in range(4000):
    nums = [random.randint(1, 30) for _ in range(random.choice([4, 6, 8]))]
    if spread(optimal_pairs_ans(nums)) != brute_spread(nums):
        counter = nums; break
print("phản ví dụ:", counter)
if counter:
    print("ghép nhỏ-lớn:", optimal_pairs_ans(counter), "spread =", spread(optimal_pairs_ans(counter)),
          "| tối ưu =", brute_spread(counter))

phản ví dụ: None


Kết luận Bài 5 (theo output: không tìm thấy phản ví dụ): ghép nhỏ–lớn **cũng** tối ưu cho hiệu max − min. Lý do (lập luận đổi chỗ trên bốn số `a ≤ b ≤ c ≤ d`): hai tổng của cách đối đầu `a + d` và `b + c` đều nằm trong đoạn `[x, y]` tạo bởi hai tổng của **mọi** cách ghép khác — với `(a, b), (c, d)`: `a + b ≤ a + d, b + c ≤ c + d`; với `(a, c), (b, d)`: `a + c ≤ a + d, b + c ≤ b + d`. Do đó đổi hai cặp "không đối đầu" thành "đối đầu" **không làm tăng** max và **không làm giảm** min của toàn bộ — hiệu không tăng. Lặp lại tới khi mọi cặp đều đối đầu.

Bài học: khi một tham lam tối ưu cho cả max lẫn min, hãy kiểm tra xem phép đổi chỗ có "thu hẹp" khoảng giá trị không — nếu có, nó tối ưu cho mọi mục tiêu đơn điệu theo max và min.

## 20. Template nhớ nhanh + Checklist

```python
# Minimize max pair sum: sắp, ghép nhỏ nhất với lớn nhất
a = sorted(nums); n = len(a)
return max(a[i] + a[n - 1 - i] for i in range(n // 2))

# Khuôn nhị phân trên đáp án (minimize the maximum)
lo, hi = LOW, HIGH
while lo < hi:
    mid = (lo + hi) // 2
    if feasible(mid): hi = mid
    else: lo = mid + 1
return lo
```

**Checklist trước khi submit:**
- [ ] Sắp xếp trước khi ghép.
- [ ] Ghép **đối đầu** (`i` với `n−1−i`), không ghép kề nhau.
- [ ] Lấy max trên **mọi** cặp, không chỉ cặp đầu.
- [ ] Nhị phân: `hi = mid` khi khả thi.
- [ ] Thử mảng toàn bằng nhau và `n = 2`.